# NB03 — Embedding & Retrieval Benchmark

**목표:** NB02에서 만든 `chunks_naive.json`과 `chunks_legal_aware.json`을 같은 임베딩 모델과 같은 Q1–Q10 질의로 검색해, 어떤 청킹 방식이 필요한 법적 근거를 더 잘 찾아오는지 비교한다.

> 이 노트북은 아직 답변을 생성하지 않는다. LLM을 연결하기 전에 **검색 품질만** 평가한다. 점수가 높아도 실제 관련 근거가 맞는지는 검색 결과를 직접 확인한다.

## 실행 순서
1. NB02 v2를 **Run All**해 `chunks_naive.json`, `chunks_legal_aware.json`을 새로 생성한다.
2. 이 노트북과 두 JSON 파일을 같은 작업 폴더에 둔다.
3. 셀을 위에서 아래로 실행한다. 첫 실행에서는 임베딩 모델 다운로드에 시간이 걸릴 수 있다.
4. Q1–Q10의 Top-5 검색 결과와 비교표를 검토한다.

**모델:** `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`  
영어 법령과 한국어 질의를 함께 다룰 수 있는 다국어 문장 임베딩 모델이다. 이번 단계에서는 로컬 임베딩과 정확 코사인 유사도 검색을 사용하므로 별도 벡터 DB나 유료 API가 필요하지 않다.

**주의:** 아래의 expected evidence는 검색 결과를 점검하기 위한 초기 평가 기준이다. 이를 정답 그 자체로 취급하지 말고, Q1–Q10의 고정된 internal project 기준표와 실제 법령 원문으로 검토한다.


In [1]:
# 필요한 라이브러리 설치
%pip install -q sentence-transformers pandas numpy


Note: you may need to restart the kernel to use updated packages.


In [3]:
from pathlib import Path
import json
import re
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

DATA_DIR = Path("data")
NAIVE_PATH = DATA_DIR / "chunks_naive.json"
LEGAL_PATH = DATA_DIR / "chunks_legal_aware.json"

for path in [NAIVE_PATH, LEGAL_PATH]:
    if not path.exists():
        raise FileNotFoundError(
            f"{path} 파일을 찾을 수 없습니다. NB02 v2를 Run All한 뒤 JSON 파일을 이 노트북과 같은 폴더에 두세요."
        )

with NAIVE_PATH.open(encoding="utf-8") as f:
    naive_chunks = json.load(f)
with LEGAL_PATH.open(encoding="utf-8") as f:
    legal_chunks = json.load(f)

print(f"Naive chunks: {len(naive_chunks):,}")
print(f"Legal-aware chunks: {len(legal_chunks):,}")
print("Naive sample keys:", list(naive_chunks[0].keys()))
print("Legal-aware sample keys:", list(legal_chunks[0].keys()))

# v2 QA guard: stop if the consolidated Turkish dried-fig entry is not present as one complete logical chunk.
turkey_chunks = [
    c for c in legal_chunks
    if "2019_1793" in c.get("file", "")
    and "Türkiye" in c.get("text", "")
    and "dried figs" in c.get("text", "").lower()
    and "ochratoxin" in c.get("text", "").lower()
    and "30%" in c.get("text", "")
]
print(f"Complete Türkiye dried-fig logical chunks detected: {len(turkey_chunks)}")
if len(turkey_chunks) != 1:
    print("WARNING: Expected exactly one complete Türkiye dried-fig logical chunk. Check that NB02 v2 was rerun and the JSON output was overwritten.")
else:
    print("PASS: Türkiye dried-fig scope, hazards, and 30% check frequency are grouped in one chunk.")


Naive chunks: 342
Legal-aware chunks: 1,383
Naive sample keys: ['chunk_id', 'strategy', 'file', 'regulation_title', 'document_type', 'version_date', 'canonical_url', 'location', 'source_type', 'text', 'char_count']
Legal-aware sample keys: ['chunk_id', 'strategy', 'file', 'regulation_title', 'document_type', 'version_date', 'canonical_url', 'source_type', 'location', 'article_id', 'article', 'heading', 'table_index', 'row_index', 'text', 'char_count']
Complete Türkiye dried-fig logical chunks detected: 1
PASS: Türkiye dried-fig scope, hazards, and 30% check frequency are grouped in one chunk.


## 1. Q1–Q10 검색 질의와 초기 근거 기준

질의는 우선 영어로 통일해 검색 비교의 변수를 줄인다. 법령 원문도 영어이기 때문이다. 다국어 모델이므로 한국어 질의를 추가 시험할 수 있지만, 1차 비교 점수에는 섞지 않는다.

`expected_files`와 `expected_terms`는 관련 근거를 Top-k에서 찾았는지 대략 확인하기 위한 기준이다. 한 질문에 여러 법령이 관련될 수 있으므로, 검색 결과를 읽고 최종 판단한다.


In [4]:
queries = [
    {
        "qid": "Q1",
        "question": "What is the EU legal basis for maximum levels of aflatoxins in dried figs?",
        "expected_files": ["EU_2023_915"],
        "expected_terms": ["1.1.3", "aflatoxin"],
        "why": "Regulation (EU) 2023/915, Annex I, dried figs entry"
    },
    {
        "qid": "Q2",
        "question": "What are the maximum levels for aflatoxin B1 and total aflatoxins in dried figs?",
        "expected_files": ["EU_2023_915"],
        "expected_terms": ["6,0", "10,0"],
        "why": "Dried figs row: B1 6.0 μg/kg; total aflatoxins 10.0 μg/kg"
    },
    {
        "qid": "Q3",
        "question": "How does the 80 percent rule for dried figs apply, and what does Article 3 say?",
        "expected_files": ["EU_2023_915"],
        "expected_terms": ["80", "Article 3"],
        "why": "Annex I note and Article 3(1)–(2)"
    },
    {
        "qid": "Q4",
        "question": "For dried figs originating in Türkiye, which hazards and what frequency of identity and physical checks apply under the increased official controls regime?",
        "expected_files": ["EU_2019_1793"],
        "expected_terms": ["aflatoxin", "ochratoxin", "30%"],
        "why": "Annex II, Section 1, Türkiye dried-fig product group"
    },
    {
        "qid": "Q5",
        "question": "What prior notification and border control requirements apply to these consignments, including CHED and the timing of notification?",
        "expected_files": ["EU_2017_625", "EU_2019__1013", "EU_2019_1793"],
        "expected_terms": ["CHED", "working day"],
        "why": "Official controls / CHED, prior notification and relevant implementing rules"
    },
    {
        "qid": "Q6",
        "question": "What official certificate, sampling and laboratory requirements apply to consignments subject to the special import conditions?",
        "expected_files": ["EU_2019_1793"],
        "expected_terms": ["certificate", "sampling"],
        "why": "Regulation (EU) 2019/1793, Articles 9–11"
    },
    {
        "qid": "Q7",
        "question": "What are the food business operator's responsibilities for unsafe food, traceability, withdrawal and informing authorities?",
        "expected_files": ["EU_2002_178"],
        "expected_terms": ["Article 17", "Article 18", "Article 19"],
        "why": "Regulation (EC) No 178/2002, Articles 14 and 17–19"
    },
    {
        "qid": "Q8",
        "question": "What can competent authorities do when a consignment does not comply with official import controls?",
        "expected_files": ["EU_2017_625"],
        "expected_terms": ["Article 66"],
        "why": "Regulation (EU) 2017/625, Article 66 and related measures"
    },
    {
        "qid": "Q9",
        "question": "Which exact legal provisions should be cited to support the assessment of dried figs from Türkiye for aflatoxins and ochratoxin A?",
        "expected_files": ["EU_2023_915", "EU_2019_1793"],
        "expected_terms": ["Annex", "aflatoxin"],
        "why": "Both contaminant maximum levels and enhanced import controls must be sourced"
    },
    {
        "qid": "Q10",
        "question": "How should the current applicable version and amendments of the relevant EU regulations be verified before giving regulatory advice?",
        "expected_files": ["EU_2023_915", "EU_2019_1793"],
        "expected_terms": ["2026"],
        "why": "Version metadata and source URL should be checked; static retrieval can become stale"
    },
]

queries_df = pd.DataFrame([
    {"qid": q["qid"], "question": q["question"], "initial_expected_evidence": q["why"]}
    for q in queries
])
display(queries_df)


,qid,question,initial_expected_evidence
0,Q1,What is the EU legal basis for maximum levels ...,"Regulation (EU) 2023/915, Annex I, dried figs ..."
1,Q2,What are the maximum levels for aflatoxin B1 a...,Dried figs row: B1 6.0 μg/kg; total aflatoxins...
2,Q3,How does the 80 percent rule for dried figs ap...,Annex I note and Article 3(1)–(2)
3,Q4,"For dried figs originating in Türkiye, which h...","Annex II, Section 1, Türkiye dried-fig product..."
4,Q5,What prior notification and border control req...,"Official controls / CHED, prior notification a..."
5,Q6,"What official certificate, sampling and labora...","Regulation (EU) 2019/1793, Articles 9–11"
6,Q7,What are the food business operator's responsi...,"Regulation (EC) No 178/2002, Articles 14 and 1..."
7,Q8,What can competent authorities do when a consi...,"Regulation (EU) 2017/625, Article 66 and relat..."
8,Q9,Which exact legal provisions should be cited t...,Both contaminant maximum levels and enhanced i...
9,Q10,How should the current applicable version and ...,Version metadata and source URL should be chec...


## 2. 모델 로드 및 임베딩 생성

두 청킹 방식 모두 **동일한 모델**을 사용한다. 이 비교에서 달라지는 것은 청킹 전략뿐이다. 임베딩은 한 번 계산하고, 이후 코사인 유사도로 Top-k를 구한다.


In [5]:
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
model = SentenceTransformer(MODEL_NAME)
print("Loaded embedding model:", MODEL_NAME)

def chunk_text(chunk):
    # Legal-aware chunks include location metadata; add it to the text the retriever sees.
    metadata_parts = [
        chunk.get("regulation_title", ""),
        chunk.get("location", ""),
        chunk.get("article", ""),
        chunk.get("heading", ""),
    ]
    metadata = " | ".join(str(x) for x in metadata_parts if x)
    return (metadata + "\n" + chunk.get("text", "")).strip()

naive_texts = [chunk_text(c) for c in naive_chunks]
legal_texts = [chunk_text(c) for c in legal_chunks]
query_texts = [q["question"] for q in queries]

naive_embeddings = model.encode(
    naive_texts, convert_to_numpy=True, normalize_embeddings=True,
    show_progress_bar=True
)
legal_embeddings = model.encode(
    legal_texts, convert_to_numpy=True, normalize_embeddings=True,
    show_progress_bar=True
)
query_embeddings = model.encode(
    query_texts, convert_to_numpy=True, normalize_embeddings=True,
    show_progress_bar=False
)

print("Embedding shapes:")
print("  queries:", query_embeddings.shape)
print("  naive:  ", naive_embeddings.shape)
print("  legal:  ", legal_embeddings.shape)


'(ProtocolError('Connection aborted.', ConnectionResetError(10054, '현재 연결은 원격 호스트에 의해 강제로 끊겼습니다', None, 10054, None)), '(Request ID: 96d74d69-94ce-44d2-9613-ea802eb3bd7e)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].
'(ProtocolError('Connection aborted.', ConnectionResetError(10054, '현재 연결은 원격 호스트에 의해 강제로 끊겼습니다', None, 10054, None)), '(Request ID: 4455e57f-c772-42a7-a429-e6de4ed8c2f6)')' thrown while requesting HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/e8f8c211226b894fcb81acc59f3b34ba3efd5f42/modules.json
Retrying in 1s [Retry 1/5].
'(ProtocolError('Connection aborted.', ConnectionResetError(10054, '현재 연결은 원격 호스트에 의해 강제로 끊겼습니다', None, 10054, None)), '(Request ID: b31d50a8-0395-4dcb-ae8f-88cf1d6175bb)')' thrown while requesting HEAD https://huggingface.co/api/resolve-cache/models/sentence-trans

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

c:\Users\SAMSUNG\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\SAMSUNG\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
'(ProtocolError('Connection aborted.', ConnectionResetError(10054, '현재 연결은 원격 호스

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loaded embedding model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


Batches:   0%|          | 0/11 [00:00<?, ?it/s]

Batches:   0%|          | 0/44 [00:00<?, ?it/s]

Embedding shapes:
  queries: (10, 384)
  naive:   (342, 384)
  legal:   (1383, 384)


## 3. Top-5 검색 결과

검색 점수는 코사인 유사도다. 점수는 관련성의 확률이 아니며, 서로 다른 질의 사이의 절대 점수 비교보다 같은 질의에서 나온 순위와 실제 근거의 내용을 보는 것이 중요하다.


In [6]:
TOP_K = 5

def retrieve(chunks, texts, embeddings, q, q_embedding, strategy, top_k=TOP_K):
    scores = embeddings @ q_embedding
    top_indices = np.argsort(scores)[::-1][:top_k]
    rows = []
    for rank, idx in enumerate(top_indices, start=1):
        c = chunks[int(idx)]
        rows.append({
            "qid": q["qid"],
            "strategy": strategy,
            "rank": rank,
            "score": float(scores[idx]),
            "chunk_id": c.get("chunk_id", ""),
            "file": c.get("file", ""),
            "version_date": c.get("version_date", ""),
            "location": c.get("location", ""),
            "article": c.get("article", ""),
            "heading": c.get("heading", ""),
            "text_preview": re.sub(r"\s+", " ", c.get("text", ""))[:650],
        })
    return rows

all_results = []
for i, q in enumerate(queries):
    all_results.extend(retrieve(
        naive_chunks, naive_texts, naive_embeddings, q, query_embeddings[i], "naive"
    ))
    all_results.extend(retrieve(
        legal_chunks, legal_texts, legal_embeddings, q, query_embeddings[i], "legal_aware"
    ))

results_df = pd.DataFrame(all_results)
display(results_df[results_df["rank"] <= 3][
    ["qid", "strategy", "rank", "score", "file", "location", "text_preview"]
].sort_values(["qid", "strategy", "rank"]).reset_index(drop=True))


,qid,strategy,rank,score,file,location,text_preview
0,Q1,legal_aware,1,0.604927,EU_2023_915_consolidated_2026-08-19.html,"Table 3, row 9",COMMISSION REGULATION (EU) 2023/915. Table con...
1,Q1,legal_aware,2,0.604256,EU_2023_915_consolidated_2026-08-19.html,"Table 3, row 16",COMMISSION REGULATION (EU) 2023/915. Table con...
2,Q1,legal_aware,3,0.604063,EU_2023_915_consolidated_2026-08-19.html,"Table 3, row 22",COMMISSION REGULATION (EU) 2023/915. Table con...
3,Q1,naive,1,0.598315,EU_2019_1793_consolidated_2026-06-30.html,fixed_chunk_15,7 99 39 | 05; 06 ex 2007 99 50 | 33 ex 2007 99...
4,Q1,naive,2,0.573607,EU_2023_915_consolidated_2026-08-19.html,fixed_chunk_12,into force This Regulation shall enter into fo...
5,Q1,naive,3,0.565864,EU_2023_915_consolidated_2026-08-19.html,fixed_chunk_15,"In other cases, Article 3(1) and (2) apply. 1...."
6,Q10,legal_aware,1,0.785910,EU_2017_625_consolidated_2025-01-05.html,Article 162 — Amendments to Regulation (EU) No...,"certifying products, processes and services”. ..."
7,Q10,legal_aware,2,0.776228,EU_2017_625_consolidated_2025-01-05.html,Article 147 — Relation with Regulation (EC) No...,Article 147 Relation with Regulation (EC) No 8...
8,Q10,legal_aware,3,0.746718,EU_2017_625_consolidated_2025-01-05.html,Article 161 — Amendments to Regulation (EC) No...,Article 161 Amendments to Regulation (EC) No 1...
9,Q10,naive,1,0.772222,EU_2017_625_consolidated_2025-01-05.html,fixed_chunk_66,od which is similar to the one not included wi...


## 4. 간단한 자동 점검 지표

아래 지표는 **초기 진단용 휴리스틱**이다.

- **Expected-file Hit@5:** Top-5 중 질문에 지정된 법령 파일이 하나라도 있는가?
- **Expected-term coverage@5:** Top-5 전체 텍스트에 초기 기준 용어 중 하나 이상이 있는가?
- **Both checks:** 두 조건을 모두 만족하는가?

이 기준은 완전한 relevance judgment가 아니다. 예를 들어 올바른 법령이 검색되어도 조문 번호가 해당 chunk에 없을 수 있고, 같은 단어가 나와도 질문에 답하는 근거가 아닐 수 있다. 따라서 이 점수만으로 청킹 전략의 승자를 선언하지 않는다.


In [7]:
def normalize_text(value):
    return re.sub(r"\s+", " ", str(value or "")).casefold()

metric_rows = []
for q in queries:
    for strategy in ["naive", "legal_aware"]:
        top = results_df[
            (results_df["qid"] == q["qid"]) &
            (results_df["strategy"] == strategy) &
            (results_df["rank"] <= TOP_K)
        ]
        files_text = " ".join(top["file"].fillna("").astype(str).tolist())
        body_text = " ".join(top["text_preview"].fillna("").astype(str).tolist())
        file_hit = any(expected.casefold() in files_text.casefold() for expected in q["expected_files"])
        term_hit = any(term.casefold() in normalize_text(body_text) for term in q["expected_terms"])
        metric_rows.append({
            "qid": q["qid"],
            "strategy": strategy,
            "expected_file_hit_at_5": file_hit,
            "expected_term_coverage_at_5": term_hit,
            "both_checks": file_hit and term_hit,
        })

metrics_df = pd.DataFrame(metric_rows)
display(metrics_df)

summary_df = metrics_df.groupby("strategy", as_index=False).agg(
    expected_file_hit_at_5=("expected_file_hit_at_5", "sum"),
    expected_term_coverage_at_5=("expected_term_coverage_at_5", "sum"),
    both_checks=("both_checks", "sum"),
)
summary_df["total_queries"] = len(queries)
for col in ["expected_file_hit_at_5", "expected_term_coverage_at_5", "both_checks"]:
    summary_df[col + "_rate"] = (summary_df[col] / len(queries)).round(3)
display(summary_df)


,qid,strategy,expected_file_hit_at_5,expected_term_coverage_at_5,both_checks
0,Q1,naive,True,True,True
1,Q1,legal_aware,True,True,True
2,Q2,naive,True,True,True
3,Q2,legal_aware,True,True,True
4,Q3,naive,True,True,True
5,Q3,legal_aware,True,False,False
6,Q4,naive,True,True,True
7,Q4,legal_aware,True,True,True
8,Q5,naive,True,True,True
9,Q5,legal_aware,True,True,True


,strategy,expected_file_hit_at_5,expected_term_coverage_at_5,both_checks,total_queries,expected_file_hit_at_5_rate,expected_term_coverage_at_5_rate,both_checks_rate
0,legal_aware,9,8,8,10,0.9,0.8,0.8
1,naive,9,8,8,10,0.9,0.8,0.8


## 5. 핵심 케이스 점검: Türkiye dried figs

Q4와 Q9는 특히 중요하다. `2019/1793`에서 제품 범위, aflatoxins, ochratoxin A, 30% 검사 빈도가 **한 논리적 항목 안에 함께 검색되는지** 직접 확인한다.


In [8]:
def show_query_results(qid, strategy, top_k=TOP_K):
    cols = ["rank", "score", "file", "version_date", "location", "article", "text_preview"]
    subset = results_df[
        (results_df["qid"] == qid) &
        (results_df["strategy"] == strategy) &
        (results_df["rank"] <= top_k)
    ][cols]
    print(f"\n{qid} — {strategy}")
    display(subset.reset_index(drop=True))

show_query_results("Q4", "naive")
show_query_results("Q4", "legal_aware")
show_query_results("Q9", "legal_aware")



Q4 — naive


,rank,score,file,version_date,location,article,text_preview
0,1,0.559349,EU_2019_1793_consolidated_2026-06-30.html,2026-06-30,fixed_chunk_19,,ARIC sub-division | Hazard | Frequency of iden...
1,2,0.535819,EU_2017_625_consolidated_2025-01-05.html,2025-01-05,fixed_chunk_32,,to recognised uniform hazards and risks which ...
2,3,0.529334,EU_2017_625_consolidated_2025-01-05.html,2025-01-05,fixed_chunk_80,,red to in Article 145(2). 4. The Commission sh...
3,4,0.528879,EU_2017_625_consolidated_2025-01-05.html,2025-01-05,fixed_chunk_73,,o verify compliance with the applicable requir...
4,5,0.527597,EU_2019_1793_consolidated_2026-06-30.html,2026-06-30,fixed_chunk_25,,Union | Food and feed (intended use) | CN code...



Q4 — legal_aware


,rank,score,file,version_date,location,article,text_preview
0,1,0.623495,EU_2019_1793_consolidated_2026-06-30.html,2026-06-30,"Annex II, Section 1, Row 14 — Türkiye dried-fi...",None,COMMISSION IMPLEMENTING REGULATION (EU) 2019/1...
1,2,0.538861,EU_2017_625_consolidated_2025-01-05.html,2025-01-05,"Article 54 — Frequency of documentary checks, ...",Article 54,tific assessments; and (vi) any other informat...
2,3,0.517378,EU_2019_1793_consolidated_2026-06-30.html,2026-06-30,"Table 5, row 169",None,COMMISSION IMPLEMENTING REGULATION (EU) 2019/1...
3,4,0.503277,EU_2019_1793_consolidated_2026-06-30.html,2026-06-30,Article 8 — Frequency of identity checks and p...,Article 8,Article 8 Frequency of identity checks and phy...
4,5,0.497854,EU_2017_625_consolidated_2025-01-05.html,2025-01-05,Article 18 — Specific rules on official contro...,Article 18,"consumption, the health mark shall be applied ..."



Q9 — legal_aware


,rank,score,file,version_date,location,article,text_preview
0,1,0.644034,EU_2019_1793_consolidated_2026-06-30.html,2026-06-30,"Annex II, Section 1, Row 14 — Türkiye dried-fi...",None,COMMISSION IMPLEMENTING REGULATION (EU) 2019/1...
1,2,0.535437,EU_2023_915_consolidated_2026-08-19.html,2026-08-19,"Table 4, row 3",None,COMMISSION REGULATION (EU) 2023/915. Table con...
2,3,0.534743,EU_2023_915_consolidated_2026-08-19.html,2026-08-19,"Table 4, row 7",None,COMMISSION REGULATION (EU) 2023/915. Table con...
3,4,0.532563,EU_2023_915_consolidated_2026-08-19.html,2026-08-19,"Table 4, row 19",None,COMMISSION REGULATION (EU) 2023/915. Table con...
4,5,0.523682,EU_2023_915_consolidated_2026-08-19.html,2026-08-19,"Table 4, row 6",None,COMMISSION REGULATION (EU) 2023/915. Table con...


## 6. 검색 결과 저장

저장된 결과를 나중에 NB04의 RAG 답변 평가와 연결할 수 있다. `retrieval_results_nb03.csv`는 사람이 검토할 결과이며, `retrieval_metrics_nb03.csv`는 초기 휴리스틱 지표다.


In [9]:
results_path = DATA_DIR / "retrieval_results_nb03.csv"
metrics_path = DATA_DIR / "retrieval_metrics_nb03.csv"
summary_path = DATA_DIR / "retrieval_summary_nb03.csv"

results_df.to_csv(results_path, index=False, encoding="utf-8-sig")
metrics_df.to_csv(metrics_path, index=False, encoding="utf-8-sig")
summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")

print("Saved:")
print(" -", results_path)
print(" -", metrics_path)
print(" -", summary_path)


Saved:
 - data\retrieval_results_nb03.csv
 - data\retrieval_metrics_nb03.csv
 - data\retrieval_summary_nb03.csv


## 7. 사람이 확인할 체크리스트

아래 항목을 Q1–Q10 각각에 대해 검토한 후, 별도 회고/표에 기록한다.

- [ ] Top-5에 질문에 답하는 실제 법령 근거가 있는가?
- [ ] 조문/부속서/표의 위치가 식별되는가?
- [ ] 한 청크에 답변에 필요한 핵심 조건과 예외가 함께 들어 있는가?
- [ ] 법령 제목, 버전 날짜, 출처 URL 등 추적 정보가 유지되는가?
- [ ] 같은 질의에서 naive와 legal-aware 중 어느 쪽이 더 적합한 근거를 상위에 가져왔는가?
- [ ] 근거가 누락되었을 때 청킹 문제인지, 임베딩/질의 문제인지, 원천 파싱 문제인지 구분했는가?

### NB03 완료 기준
1. 두 청킹 방식 모두 Q1–Q10의 Top-5를 생성했다.
2. 자동 지표와 Top-5 원문을 함께 검토했다.
3. 핵심 Q4/Q9에서 Türkiye dried-fig 항목의 범위·위험요소·30% 빈도가 한 청크에 보존되고 검색되는지 확인했다.
4. 검색 실패 사례를 기록한 뒤에만 NB04(LLM 기반 답변 생성)로 넘어간다.

**해석 원칙:** legal-aware가 항상 이긴다고 미리 가정하지 않는다. 검색 결과의 근거 정확성, 문맥 완결성, 출처 추적 가능성을 보고 판단한다.
